# Dataset formatting

This code will create a `dataset.csv` that contains a summary of the previously collected datasets stored in `../datasets/`. The resulting csv file will then be used for further data preprocessing.

In [1]:
# Imports

import os
import shutil as sh
from dataclasses import dataclass
import pandas as pd
import numpy as np
from PIL import Image

In [2]:
cwd: str = os.getcwd()
rel_datasets_path: str = "../datasets"
abs_dataset_path: str = os.path.join(cwd, rel_datasets_path)

def copy_dataset(dataset: str, path: str = abs_dataset_path) -> None:
    dir_path = os.path.join(path, dataset)
    dst_path = os.path.join(cwd, dataset)

    sh.copytree(dir_path, dst_path)

## CKplus dataset

In [3]:
dataset: str = "CKplus"
copy_dataset(dataset)

In [4]:
# Change column labels to lowercase
file = os.listdir(dataset)[0]
filepath = os.path.join(dataset, file)

df = pd.read_csv(filepath)
df.columns = df.columns.str.lower()

In [5]:
# drop emotions Neutral and contempt since they are not needed
mask = df["emotion"] > 5
df = df.drop(df.loc[mask].index)
df = df.reset_index(drop = True)

In [6]:
# Drop the usage column. A split will later be created by the group
df = df.drop("usage", axis = 1)

In [7]:
# Each image will be converted to an image and saved to a new collection directory
collection_dir = os.path.join(cwd, "dataset")
os.mkdir(collection_dir)
num_rows = df.shape[0]

In [8]:
arr = []
images = df["pixels"].tolist()

for img in images:
    img = list(map(int, img.split()))
    arr.append(img)

arr = np.array(arr)

df = df.drop("pixels", axis = 1)

In [9]:
files = []
for i in range(num_rows):
    filename: str = f"CKPlus_{i + 1}.png"
    filepath = os.path.join(collection_dir, filename)

    img:np.ndarray = arr[i].reshape(48, 48)
    img = Image.fromarray(img.astype(np.uint8))
    img.save(filepath)

    files.append(filename)
df["filename"] = files

In [10]:
df

,emotion,filename
0,3,CKPlus_1.png
1,3,CKPlus_2.png
2,3,CKPlus_3.png
3,3,CKPlus_4.png
4,3,CKPlus_5.png
...,...,...
304,5,CKPlus_305.png
305,5,CKPlus_306.png
306,5,CKPlus_307.png
307,5,CKPlus_308.png


In [11]:
df.to_csv("dataset.csv", index = False)

In [12]:
sh.rmtree(dataset)

## FERPlus

In [13]:
emotion_map = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "sad": 4,
    "suprise": 5,
}

@dataclass
class Entry:
    emotion: int
    filename: str

type DataList = list[Entry]

dataset: str = "FERPlus"
copy_dataset(dataset)

In [14]:
data_dir = os.path.join(cwd, dataset)
os.chdir(data_dir)
for dir in os.listdir(data_dir):
    os.chdir(dir)
    sub_dirs = os.listdir(os.getcwd()) # These are the emotion directories
    if "contempt" in sub_dirs:
        sh.rmtree("contempt")
    if "neutral" in sub_dirs:
        sh.rmtree("neutral")
    os.chdir(data_dir)

os.chdir(cwd)

In [15]:
collection_dir = os.path.join(data_dir, "collection")
os.mkdir(collection_dir)

test_dir = os.listdir(data_dir)[0]
category_dir_1 = os.path.join(data_dir, test_dir)
categories = os.listdir(category_dir_1)

for category in categories:
    os.mkdir(os.path.join(collection_dir, category))

In [16]:
# Here we save all images into the collection dir
file_mappings = []
for dir in os.listdir(data_dir):
    
    split = os.path.join(data_dir, dir)

    for emotion in os.listdir(split):

        current = os.path.join(split, emotion)

        for image in os.listdir(current):
            count = len(file_mappings) + 1
            file_mappings.append(Entry(emotion_map[os.path.basename(current)], f"FERPlus_{count}.png"))

            src = os.path.join(current, image)
            dst = os.path.join(cwd, "dataset", f"FERPlus_{count}.png")
            sh.copy(src, dst)

In [17]:
df1 = pd.read_csv("dataset.csv")
df2 = pd.DataFrame(file_mappings)

combined = pd.concat([df1, df2], ignore_index = True)
combined.to_csv("dataset.csv", index = False)

In [18]:
sh.rmtree(dataset)

## AffectNet

In [19]:
dataset: str = "AffectNet"
copy_dataset(dataset)

In [20]:
emotion_map = {
    "anger": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "sad": 4,
    "surprise": 5,
}

In [21]:
AffectNet_path = os.path.join(cwd, dataset)
labels_path = os.path.join(AffectNet_path, "labels.csv")

df = pd.read_csv(labels_path)
df = df.drop(df.columns.tolist()[0], axis=1)
df = df.drop(df.columns.tolist()[-1], axis=1)

In [22]:
unwanted_label_mask = df["label"] == "neutral"
df = df.drop(df.loc[unwanted_label_mask].index)
unwanted_label_mask = df["label"] == "contempt"
df = df.drop(df.loc[unwanted_label_mask].index)

df["label"] = df["label"].replace(emotion_map)
df.to_csv(labels_path, index = False)

/tmp/ipykernel_4179/3725023767.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"] = df["label"].replace(emotion_map)


In [23]:
file_mappings = []

for data in os.listdir(AffectNet_path):

    if data == "labels.csv":
        continue
    
    split = os.path.join(AffectNet_path, data)

    for emotion in os.listdir(split):

        current = os.path.join(split, emotion)
        
        emotion = emotion.lower()
        for image in os.listdir(current):

            src = os.path.join(emotion, image)
            if src in df["pth"].tolist():
                count = len(file_mappings) + 1
                label = df.loc[df["pth"] == src, "label"].iloc[0]
                filetype = ".jpg" if src.endswith(".jpg") else ".png"
                file_mappings.append(Entry(label, f"AffectNet_{count}{filetype}"))
                sh.copy(os.path.join(current, image), os.path.join(cwd, "dataset", f"AffectNet_{count}{filetype}"))
            

In [24]:
df = pd.DataFrame(file_mappings)
df2 = pd.read_csv("dataset.csv")

combined = pd.concat([df2, df], ignore_index=True)
combined.to_csv("dataset.csv", index=False)

In [25]:
sh.rmtree(dataset)